In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [7]:
ABSA_FILE = 'absa_embeddings.npy'
EMOTION_FILE = 'emotion_embeddings.npy'
LABEL_FILE = 'dataset/cleaned_data_for_training.csv'
OUTPUT_DIR = 'processed_data'

def prepare_data_splits_final():
    print("1. Loading Data...")
    X_absa = np.load(ABSA_FILE)
    X_emo = np.load(EMOTION_FILE)
    df = pd.read_csv(LABEL_FILE)
    
    y_target = df['deceptive_flag'].values
    
    le_pol = LabelEncoder()
    y_polarity = le_pol.fit_transform(df['polarity']) 
    print(f"   Polarity Encoding: {dict(zip(le_pol.classes_, le_pol.transform(le_pol.classes_)))}")

    df['stratify_group'] = df['deceptive_flag'].astype(str) + "_" + df['polarity'].astype(str)
    le_group = LabelEncoder()
    y_groups = le_group.fit_transform(df['stratify_group'])
    
    print("\n2. Performing Stratified 85-15 Split...")
    indices = np.arange(len(df))
    
    idx_train, idx_test, _, _ = train_test_split(
        indices, 
        y_groups,
        test_size=0.15, 
        stratify=y_groups, 
        random_state=42
    )
    
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
    print(f"\n3. Saving files to '{OUTPUT_DIR}/'...")
    
    np.save(f'{OUTPUT_DIR}/X_absa_train.npy', X_absa[idx_train])
    np.save(f'{OUTPUT_DIR}/X_absa_test.npy',  X_absa[idx_test])
    np.save(f'{OUTPUT_DIR}/X_emo_train.npy',  X_emo[idx_train])
    np.save(f'{OUTPUT_DIR}/X_emo_test.npy',   X_emo[idx_test])
    
    np.save(f'{OUTPUT_DIR}/y_train.npy', y_target[idx_train])
    np.save(f'{OUTPUT_DIR}/y_test.npy',  y_target[idx_test])
    
    np.save(f'{OUTPUT_DIR}/groups_train.npy', y_groups[idx_train])
    
    np.save(f'{OUTPUT_DIR}/polarity_train.npy', y_polarity[idx_train])
    np.save(f'{OUTPUT_DIR}/polarity_test.npy',  y_polarity[idx_test])
    
    print("✅ SUCCESS: All files saved.")

if __name__ == "__main__":
    prepare_data_splits_final()

1. Loading Data...
   Polarity Encoding: {'negative': np.int64(0), 'positive': np.int64(1)}

2. Performing Stratified 85-15 Split...

3. Saving files to 'processed_data/'...
✅ SUCCESS: All files saved.


In [ ]:
#testing part

In [9]:
#testing
absa = np.load('processed_data/X_absa_train.npy')
emo  = np.load('processed_data/X_emo_train.npy')
y    = np.load('processed_data/y_train.npy')
absa_test = np.load('processed_data/X_absa_test.npy')
emo_test  = np.load('processed_data/X_emo_test.npy')
y_test    = np.load('processed_data/y_test.npy')

print(f"ABSA Rows:    {absa.shape[0]}")
print(f"Emotion Rows: {emo.shape[0]}")
print(f"Label Rows:   {y.shape[0]}")
print(f"Test ABSA Rows:    {absa_test.shape[0]}")
print(f"Test Emotion Rows: {emo_test.shape[0]}")
print(f"Test Label Rows:   {y_test.shape[0]}")

if absa.shape[0] == emo.shape[0] == y.shape[0]:
    print("✅ All files are aligned.")
else:
    print("❌ ERROR: Mismatch detected.")

ABSA Rows:    1360
Emotion Rows: 1360
Label Rows:   1360
Test ABSA Rows:    240
Test Emotion Rows: 240
Test Label Rows:   240
✅ All files are aligned.


In [11]:
print("Loading Test Data...")
X_absa = np.load('processed_data/X_absa_test.npy')
X_emo  = np.load('processed_data/X_emo_test.npy')
y      = np.load('processed_data/y_test.npy')
groups_train = np.load('processed_data/groups_train.npy')
polarity_train = np.load('processed_data/polarity_train.npy')

print("\n" + "="*50)
print("   DATA INSPECTION REPORT")
print("="*50)

print(f"\n1. ABSA Features (X_absa_test)")
print(f"   Shape: {X_absa.shape}  (Reviews x Dimensions)")
print(f"   Data Type: {X_absa.dtype}")
print("   First Review, First 10 Numbers:")
print("   ", X_absa[0][:10]) 

print(f"\n2. Emotion Features (X_emo_test)")
print(f"   Shape: {X_emo.shape}")
print("   First Review, First 10 Numbers:")
print("   ", X_emo[0][:10])

print(f"\n3. Labels (y_test)")
print(f"   Shape: {y.shape}")
print("   First 10 Labels (0=Genuine, 1=Fraud):")
print("   ", y[:10])

print(f"\n4. Labels (groups_train)")
print(f"   Shape: {y.shape}")
print("   First 10 Labels:")
print("   ", groups_train[:10])

print(f"\n5. Labels (polarity_train)")
print(f"   Shape: {y.shape}")
print("   First 10 Labels:")
print("   ", polarity_train[:10])

print("\n" + "="*50)

Loading Test Data...

   DATA INSPECTION REPORT

1. ABSA Features (X_absa_test)
   Shape: (240, 768)  (Reviews x Dimensions)
   Data Type: float32
   First Review, First 10 Numbers:
    [-0.80807    -0.51595026  0.18292399 -0.35160464 -1.410634   -0.0701044
 -0.16797695 -0.06875209  0.56173974 -0.25006095]

2. Emotion Features (X_emo_test)
   Shape: (240, 768)
   First Review, First 10 Numbers:
    [-0.30283755  1.5857916   0.00195295 -0.62498564 -0.6795538   1.3812244
 -0.85561544 -1.4546875   0.9200622  -0.2894977 ]

3. Labels (y_test)
   Shape: (240,)
   First 10 Labels (0=Genuine, 1=Fraud):
    [0 0 0 0 1 1 0 0 0 0]

4. Labels (groups_train)
   Shape: (240,)
   First 10 Labels:
    [1 3 0 0 3 1 0 3 3 2]

5. Labels (polarity_train)
   Shape: (240,)
   First 10 Labels:
    [1 1 0 0 1 1 0 1 1 0]



In [8]:

dummy_data = ['0_negative', '0_positive', '1_negative', '1_positive']
le = LabelEncoder()
encoded = le.fit_transform(dummy_data)

print("Verification Map:")
for num, name in zip(encoded, dummy_data):
    print(f"  {num} = {name}")

Verification Map:
  0 = 0_negative
  1 = 0_positive
  2 = 1_negative
  3 = 1_positive


In [10]:

DATA_DIR = 'processed_data'

def inspect_data_final():
    print("Loading ALL data files for inspection...")
    
    X_absa_train = np.load(os.path.join(DATA_DIR, 'X_absa_train.npy'))
    y_train      = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
    groups_train = np.load(os.path.join(DATA_DIR, 'groups_train.npy'))
    pol_train    = np.load(os.path.join(DATA_DIR, 'polarity_train.npy'))
    
    X_absa_test  = np.load(os.path.join(DATA_DIR, 'X_absa_test.npy'))
    y_test       = np.load(os.path.join(DATA_DIR, 'y_test.npy'))
    pol_test     = np.load(os.path.join(DATA_DIR, 'polarity_test.npy'))

    print("\n" + "="*60)
    print("   DATASET AUDIT REPORT")
    print("="*60)

    # 1. TRAINING SET INSPECTION
    print(f"\n[TRAINING SET]")
    print(f"   Features Shape: {X_absa_train.shape}")
    print(f"   Labels Shape:   {y_train.shape}")
    print(f"   Groups Shape:   {groups_train.shape}")
    print(f"   Polarity Shape: {pol_train.shape}")
    
    print("\n   > INSPECTING 'GROUPS': (0 = Genuine_negative | 1 = Genuine_positive | 2 = Fraud_negative | 3 = Fraud_positive)")
    print(f"     First 10 values: {groups_train[:10]}")
    print(f"     Unique Groups Found: {np.unique(groups_train)}")
    
    print("\n   > INSPECTING 'POLARITY' (0=Neg, 1=Pos):")
    print(f"     First 10 values: {pol_train[:10]}")

    # 2. TEST SET INSPECTION
    print(f"\n" + "-"*30)
    print(f"[TEST SET]")
    print(f"   Features Shape: {X_absa_test.shape}")
    print(f"   Labels Shape:   {y_test.shape}")
    print(f"   Polarity Shape: {pol_test.shape}")

    # 3. ALIGNMENT CHECK
    print(f"\n" + "="*60)
    if (X_absa_train.shape[0] == y_train.shape[0] == groups_train.shape[0] == pol_train.shape[0]):
        print("✅ TRAINING SET ALIGNMENT CORRECT")
    else:
        print("❌ TRAINING SET ALIGNMENT FAILED")
        
    if (X_absa_test.shape[0] == y_test.shape[0] == pol_test.shape[0]):
        print("✅ TEST SET ALIGNMENT CORRECT")
    else:
        print("❌ TEST SET ALIGNMENT FAILED")
    print("="*60)

if __name__ == "__main__":
    inspect_data_final()

Loading ALL data files for inspection...

   DATASET AUDIT REPORT

[TRAINING SET]
   Features Shape: (1360, 768)
   Labels Shape:   (1360,)
   Groups Shape:   (1360,)
   Polarity Shape: (1360,)

   > INSPECTING 'GROUPS': (0 = Genuine_negative | 1 = Genuine_positive | 2 = Fraud_negative | 3 = Fraud_positive)
     First 10 values: [1 3 0 0 3 1 0 3 3 2]
     Unique Groups Found: [0 1 2 3]

   > INSPECTING 'POLARITY' (0=Neg, 1=Pos):
     First 10 values: [1 1 0 0 1 1 0 1 1 0]

------------------------------
[TEST SET]
   Features Shape: (240, 768)
   Labels Shape:   (240,)
   Polarity Shape: (240,)

✅ TRAINING SET ALIGNMENT CORRECT
✅ TEST SET ALIGNMENT CORRECT
